# Week 64, Day 4: Objects in C: PyObject

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w064d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import sys

ascii_text = "a" * 100
euro_text = "€" * 100
emoji_text = "😀" * 100
for s in (ascii_text, euro_text, emoji_text):
    print(len(s), (sys.getsizeof(s + s[0]) - sys.getsizeof(s)))

small, big = 2 ** 10, 2 ** 1000
print(sys.getsizeof(big) > sys.getsizeof(small))

In [ ]:
import struct, sys

pointer = struct.calcsize("P")                  # the size of a pointer on this machine
print(pointer in (4, 8))
one, two = tuple(range(100)), tuple(range(200))
print((sys.getsizeof(two) - sys.getsizeof(one)) // 100 == pointer)    # one pointer per item

names = ["x" * 1000 for _ in range(10)]
shallow = sys.getsizeof(names)
deep = shallow + sum(sys.getsizeof(s) for s in names)
print(shallow < 1000 < deep)

In [ ]:
class Money:
    def __init__(self, pence):
        self.pence = pence
    def __add__(self, other):
        return Money(self.pence + other.pence)
    def __len__(self):
        return 3

m = Money(5) + Money(7)
print(m.pence, len(m))

m.__len__ = lambda: 99            # an instance attribute: no slot changes
print(len(m))
Money.__len__ = lambda self: 42   # setting it on the type updates the slot
print(len(m))
print(type(m).__name__, type(type(m)).__name__, type(m).__mro__[-1].__name__)

In [ ]:
import sys

bits = sys.int_info.bits_per_digit
n = 2 ** 65 + 7
digits = []
value = n
while value:
    digits.append(value & ((1 << bits) - 1))     # the lowest digit
    value >>= bits
print(bits, digits if bits == 30 else "(not a 30-bit build)")
print(sum(d << (bits * i) for i, d in enumerate(digits)) == n)

import time
for size in (1000, 100000):
    x = 7 ** size
    start = time.perf_counter()
    y = x * x
    print(size, x.bit_length() > size, time.perf_counter() - start >= 0)

In [ ]:
import sys

def bytes_per_char(s):
    return sys.getsizeof(s + s[-1]) - sys.getsizeof(s)

for text in ("plain", "café", "€uro", "hi 😀"):
    print(repr(text), max(map(ord, text)), bytes_per_char(text * 20))

items, sizes = [], []
for i in range(30):
    items.append(i)
    size = sys.getsizeof(items)
    if not sizes or size != sizes[-1][1]:
        sizes.append((len(items), size))
print(len(sizes) < 30, sizes[0][0])
print(sys.getsizeof(tuple(range(10))) < sys.getsizeof(list(range(10))))

In [ ]:
import sys

a = int("200") + 56
b = 256
c = int("1000") + 1
d = 1001
print(a is b, c == d)                # the first is True only because of the small-int cache

name = "".join(["user", "_", "id"])
print(name == "user_id", sys.intern(name) is sys.intern("user_id"))

ids = set()
for _ in range(5):
    ids.add(id(object()))            # each object dies at once; memory is reused
print(len(ids) <= 5)

> **Key idea.** Every CPython object starts with a header: a reference count and a pointer to its type (plus ob_size for variable-sized objects); containers hold pointers, and sys.getsizeof counts only the object's own memory. Type objects hold slots of C functions (tp_hash, nb_add, sq_length, ...); defining a special method fills the slot, which is why special methods live on the type. ints are arrays of 30-bit digits with a sign, so they have no size limit. Strings use 1, 2 or 4 bytes per character depending on their widest character; lists over-allocate so append is amortised O(1); tuples store items inline. Small ints, interned strings, free lists, immortal objects and pymalloc avoid allocation, and freed memory (and ids) are reused.

### Exercise: Big integers in digits

Write `to_digits(n, bits=30)`: the list of base-2^bits digits of a non-negative integer, least significant first (`[0]` for zero), using shifts and masks; `from_digits(digits, bits=30)`: the integer back; and `add_digits(a, b, bits=30)`: add two digit lists **digit by digit** with carrying, as CPython does, without converting them to integers. Don't use `divmod` or `%` on the whole number for `to_digits`.

In [ ]:
def to_digits(n, bits=30):
    ___

def from_digits(digits, bits=30):
    ___

def add_digits(a, b, bits=30):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_digits():\n    \"arbitrary precision\"\n    n = 2 ** 65 + 7\n    assert to_digits(n) == [7, 0, 32] and from_digits([7, 0, 32]) == n\n    assert to_digits(0) == [0] and to_digits(255, bits=4) == [15, 15] and from_digits([15, 15], 4) == 255\n    big = 3 ** 200\n    assert from_digits(to_digits(big)) == big\n    for x, y in ((2 ** 30 - 1, 1), (big, big), (12345, 2 ** 90), (0, 0)):\n        assert from_digits(add_digits(to_digits(x), to_digits(y))) == x + y\n    assert add_digits([15], [1], bits=4) == [0, 1]")

### Exercise: Bytes per character

Write `expected_width(s)`: the bytes per character CPython should use for the string, by PEP 393's rule (1 if every code point is below 256, 2 if below 65,536, otherwise 4; 1 for the empty string). Then `measured_width(s)`: measure it with `sys.getsizeof`, comparing a string with one more of its own characters (for the empty string, return 1). The two must agree.

In [ ]:
import sys

def expected_width(s):
    ___

def measured_width(s):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_strkind():\n    \"compact strings\"\n    for text, width in ((\"hello\", 1), (\"caf\u00e9\", 1), (\"\u0175\", 2), (\"\u20ac100\", 2), (\"hi \ud83d\ude00\", 4), (\"\", 1)):\n        assert expected_width(text) == width, text\n        assert measured_width(text * 10) == width, text\n    assert measured_width(\"a\" * 5 + \"\ud83d\ude00\") == 4")

### Exercise: How often does a list grow?

Write `resize_points(n)`: append `n` items to an empty list one by one and return the list of lengths at which `sys.getsizeof` of the list **changed** (the moments CPython had to allocate a bigger array). Then `simulate_growth(n, grow)`: simulate a dynamic array starting with capacity 0, where `grow(needed)` returns the new capacity whenever an append finds it full; return `(resizes, copies)`, the number of reallocations and the total number of items copied into new arrays.

In [ ]:
import sys

def resize_points(n):
    ___

def simulate_growth(n, grow):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_growth():\n    \"over-allocation\"\n    points = resize_points(1000)\n    assert points[0] == 1 and 5 < len(points) < 100 and points == sorted(points)\n    gaps = [b - a for a, b in zip(points, points[1:])]\n    assert gaps[-1] > gaps[0]\n    assert simulate_growth(8, lambda need: need) == (8, 28)\n    assert simulate_growth(8, lambda need: max(1, 2 * (need - 1))) == (4, 7)\n    resizes, copies = simulate_growth(10000, lambda need: need + (need >> 3) + 6)\n    assert resizes < 100 and copies < 10 * 10000")

### Exercise: Measuring layouts

Write `item_cost(make, n=100)`: the extra bytes per item of the objects built by `make(count)`, measured as `(getsizeof(make(2 * n)) - getsizeof(make(n))) / n`. Then `deep_size(obj)`: the total `sys.getsizeof` of an object and everything reachable from it through lists, tuples, sets and dicts (keys and values), counting each distinct object once (by `id`), without recursion.

In [ ]:
import struct
import sys

def item_cost(make, n=100):
    ___

def deep_size(obj):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_pointers():\n    \"memory layouts\"\n    import struct\n    pointer = struct.calcsize(\"P\")\n    assert item_cost(lambda k: tuple(range(k))) == pointer\n    assert item_cost(lambda k: b\"x\" * k) == 1 and item_cost(lambda k: \"\u00e9\" * k) == 1 and item_cost(lambda k: \"\u20ac\" * k) == 2\n    words = [\"word\" * 50] * 3\n    assert deep_size(words) == sys.getsizeof(words) + sys.getsizeof(words[0])\n    data = {\"a\": [1, 2], \"b\": (3,)}\n    expected = sys.getsizeof(data) + sum(map(sys.getsizeof, [\"a\", \"b\", data[\"a\"], data[\"b\"], 1, 2, 3]))\n    assert deep_size(data) == expected\n    loop = []\n    loop.append(loop)\n    assert deep_size(loop) == sys.getsizeof(loop)")

### Exercise: Types with slots

Simulate CPython's type objects. Write `CType(name, base=None, **slots)`: a type with C-style slots such as `add`, `length` and `repr` (functions). `lookup(slot)` finds a slot on the type or its bases (`None` if missing). `CObject(ctype, value)` is an object with a header (`refcnt` starting at 1, and `ctype`) and a `value`. Then `binary_add(a, b)`: call `a`'s type's `add` slot with `(a, b)`, or, if it's missing, `b`'s, or raise `TypeError("unsupported operand types: X and Y")`; and `c_len(obj)`: call the `length` slot or raise `TypeError("object of type X has no len()")`.

In [ ]:
class CType:
    def __init__(self, name, base=None, **slots):
        ___

class CObject:
    def __init__(self, ctype, value):
        ___

def binary_add(a, b):
    ___

def c_len(obj):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_slots():\n    \"type slots\"\n    Int = CType(\"int\", add=lambda a, b: CObject(Int, a.value + b.value))\n    Bool = CType(\"bool\", base=Int)\n    Str = CType(\"str\", add=lambda a, b: CObject(Str, a.value + b.value), length=lambda s: len(s.value))\n    x = binary_add(CObject(Int, 2), CObject(Bool, 1))\n    assert x.value == 3 and x.ctype is Int and x.refcnt == 1\n    assert Bool.lookup(\"add\") is Int.lookup(\"add\") and Bool.lookup(\"length\") is None\n    assert c_len(CObject(Str, \"abc\")) == 3 and binary_add(CObject(Str, \"a\"), CObject(Str, \"b\")).value == \"ab\"\n    NoAdd = CType(\"thing\")\n    assert binary_add(CObject(NoAdd, 1), CObject(Int, 2)).value == 3\n    try:\n        binary_add(CObject(NoAdd, 1), CObject(NoAdd, 2))\n        assert False\n    except TypeError as e:\n        assert str(e) == \"unsupported operand types: thing and thing\"\n    try:\n        c_len(CObject(Int, 5))\n        assert False\n    except TypeError as e:\n        assert str(e) == \"object of type int has no len()\"")

### Bug hunt: Identity instead of equality

`same_total` compares two totals with `is`, which asks whether they're the same **object**. 300 is outside the small-integer cache, so two separately computed 300s are different objects and this prints `False`. Compare values so it prints `True`.

In [ ]:
def same_total(a, b):
    return sum(a) is sum(b)

print(same_total([100, 200], [150, 150]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"== for values\"\n    expect_output(\"True\")")

### Bug hunt: A size that ignores the contents

`sys.getsizeof` counts only the list object (its header and pointer array), not the long strings it points to, so the program thinks the data is small and prints `False`. Add the sizes of the strings too so it prints `True`.

In [ ]:
import sys

lines = ["x" * 5000 for _ in range(4)]
total = sys.getsizeof(lines)
print(total > 20000)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"contents count\"\n    expect_output(\"True\")")

### Bug hunt: The wrong mask

To take the lowest digit, the code uses `n & bits`, which masks with the number 30 itself instead of thirty 1-bits, so the digits come out wrong and the round trip prints `False`. Mask with `(1 << bits) - 1` so it prints `True`.

In [ ]:
def digits_of(n, bits=30):
    out = []
    while n:
        out.append(n & bits)
        n >>= bits
    return out

n = 2 ** 70 + 12345
d = digits_of(n)
print(sum(x << (30 * i) for i, x in enumerate(d)) == n)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"masks\"\n    expect_output(\"True\")")

### Puzzle: Splitting an integer into digits

Arrange the lines of a function that splits a non-negative integer into base-2^bits digits, least significant first. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the two fields of the PyObject header: ("ob_refcnt", "?")
q = ___   # bytes per character of "naïve" in CPython's compact strings
r = ___   # True or False: sys.getsizeof([big_string]) includes the string's size
s = ___   # True or False: a tuple has spare room for more items
t = ___   # the usual number of bits per digit in CPython ints

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == (\"ob_refcnt\", \"ob_type\") and q == 1 and r is False, \"p, q and r\"\n    assert s is False and t == 30, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a tiny object runtime

Simulate how CPython manages objects. Write `Runtime()` with:

- `new_int(v)`: integers from -5 to 256 come from a **cache** made when the runtime starts: the same object every time, and **immortal** (their count never changes). Others are new objects. Objects are dicts like `{"type": "int", "value": v, "refcnt": 1, "addr": n}`.
- `new_tuple(items)`: a tuple object holding references to the given objects (each item's count goes up by one, except immortal ones).
- `incref(obj)` / `decref(obj)`: change the count (no effect on immortal objects). At zero, the object is **freed**: its items are decref'd, and its address goes back to be reused.
- Addresses: each new object gets the **lowest free address** (start at 1000, step 1; the cache uses 1 to 262 first). A freed tuple of length 1 to 3 goes on a **free list** for its length (at most 2 per length), keeping its address: the next `new_tuple` of that length reuses the most recently freed one instead of taking a new address.
- `live()`: the number of live non-immortal objects; `freed`: a list of addresses in the order objects were freed.

In [ ]:
class Runtime:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_ints():\n    \"the small-int cache\"\n    rt = Runtime()\n    a, b = rt.new_int(7), rt.new_int(7)\n    assert a is b and a[\"addr\"] == 1 + 7 + 5 and rt.new_int(-5)[\"addr\"] == 1\n    before = a[\"refcnt\"]\n    rt.incref(a)\n    rt.decref(a)\n    rt.decref(a)\n    assert a[\"refcnt\"] == before and rt.live() == 0\n    big = rt.new_int(1000)\n    other = rt.new_int(1000)\n    assert big is not other and big[\"addr\"] == 1000 and other[\"addr\"] == 1001 and rt.live() == 2\n    rt.decref(big)\n    assert rt.freed == [1000] and rt.new_int(5000)[\"addr\"] == 1000\n\ndef test_tuples():\n    \"references, freeing and free lists\"\n    rt = Runtime()\n    x = rt.new_int(500)\n    t = rt.new_tuple([x, rt.new_int(1)])\n    assert x[\"refcnt\"] == 2 and t[\"value\"] == [x, rt.new_int(1)] and rt.live() == 2\n    rt.decref(x)\n    assert x[\"refcnt\"] == 1\n    rt.decref(t)\n    assert rt.freed == [t[\"addr\"], x[\"addr\"]] and rt.live() == 0\n    again = rt.new_tuple([rt.new_int(2), rt.new_int(3)])\n    assert again[\"addr\"] == t[\"addr\"] and again is not t and rt.live() == 1\n    y = rt.new_int(999)\n    assert y[\"addr\"] == x[\"addr\"]\n    long_tuple = rt.new_tuple([rt.new_int(i) for i in range(5)])\n    addr = long_tuple[\"addr\"]\n    rt.decref(long_tuple)\n    assert rt.new_int(777)[\"addr\"] == addr")

## Warm-up

All three strings have 100 characters, but adding one more character costs 1, 2 or 4 bytes depending on the **widest** character in the string. And a huge integer takes more memory than a small one: Python integers have no fixed size.

## The object header

In C, every Python object starts with the same header, a struct called `PyObject`:

```
typedef struct {
    Py_ssize_t ob_refcnt;      /* the reference count (Week 62) */
    PyTypeObject *ob_type;     /* a pointer to the object's type */
} PyObject;
```

Objects whose size varies (ints, tuples, bytes, and the header part of lists) use `PyVarObject`, which adds `ob_size`, the number of items. After the header comes the type's own data: a float's C `double`, a tuple's array of pointers to its items, a list's pointer to a separately allocated array.

Some consequences:

- Every C function in CPython receives and returns `PyObject *` pointers: "everything is an object" is literally true at this level, and `id(obj)` in CPython is the object's memory address.
- **Containers hold pointers**, never the objects themselves. That's why assignment never copies (Week 62), and why a tuple's size grows by one pointer per item, whatever the items are.
- `sys.getsizeof(obj)` reports the size of **this object's own memory**, header included, but **not** the objects it points to.
- In the **free-threaded** build of Python 3.13 and later (without the global interpreter lock, Week 66), the header holds more fields, to count references safely across threads.

## Type objects and slots

`ob_type` points to a **type object**, a large C struct (`PyTypeObject`) describing the type: its name, its size, its base classes, and dozens of **slots**, pointers to the C functions that implement each operation:

| slot | used by | special method |
|---|---|---|
| `tp_hash` | `hash(x)`, dict keys | `__hash__` |
| `tp_repr`, `tp_str` | `repr`, `str` | `__repr__`, `__str__` |
| `tp_call` | `x(...)` | `__call__` |
| `tp_getattro` | `x.name` | `__getattribute__` |
| `nb_add` (in `tp_as_number`) | `x + y` | `__add__` |
| `sq_length` / `mp_length` | `len(x)` | `__len__` |
| `tp_iter`, `tp_iternext` | `for` | `__iter__`, `__next__` |

When you write `a + b`, the evaluation loop calls the C function in the type's `nb_add` slot; for built-in types that's C code written directly. When **your** class defines `__add__`, CPython fills the slot with a small C function that looks up and calls your Python method. That's why special methods are looked up on the **type**, not the instance (Week 31, and last Wednesday): the slot belongs to the type object.

## Integers of any size

A Python `int` can be as large as memory allows. CPython stores it as a sign and an array of **digits** in a large base, usually 2 to the power 30 (`sys.int_info.bits_per_digit`). The number 2^65 + 7, for example, is stored as three digits, least significant first: `[7, 0, 32]`, since 2^65 = 32 x (2^30)^2.

Arithmetic works digit by digit, like long addition at school but in base 2^30, carrying between digits. Small values take one digit and are handled by fast paths; large values cost time in proportion to their number of digits (and multiplying big numbers uses Karatsuba's method). Why 30 bits and not 32? So that adding two digits, or multiplying them, fits comfortably in C's 64-bit integers without overflowing.

The same idea appears whenever numbers exceed hardware: cryptography libraries, arbitrary-precision decimals (`decimal`), and calculators.

## Strings, lists and tuples

**Strings** (PEP 393, Python 3.3 and later) are stored compactly: CPython looks at the **largest code point** in the string and picks 1 byte per character (all below 256: Latin-1, which includes ASCII), 2 bytes (all below 65,536) or 4 bytes (anything else, such as most emoji). One emoji in a long English text makes the whole string four times bigger in memory. Strings are immutable, so the choice is made once. CPython also caches each string's hash and, for ASCII strings, their UTF-8 form.

**Lists** hold a pointer to a separate array of item pointers, with spare room: when an `append` finds the array full, CPython allocates a bigger one with about an eighth extra space (the exact formula changes between versions) and copies the pointers across. Because the spare room grows with the list, `append` is **amortised O(1)** (Week 41's dynamic arrays). `sys.getsizeof` shows the allocated size, so it jumps only now and then as you append.

**Tuples** store their item pointers directly inside the object, with no spare room, since they never change size: slightly smaller and faster to create than lists.

## Caches, free lists and immortal objects

Allocating and freeing memory costs time, so CPython avoids it where it can:

- **Small integers** from -5 to 256 are created once at start-up and shared: every `5` in your program is the same object. That's why `is` sometimes seems to work on small numbers (and why you must never rely on it, Week 62).
- **Interned strings**: identifiers, attribute names and many short literals are kept in a table so that equal names are the same object, making dict lookups by name faster. `sys.intern(s)` adds a string yourself.
- **Free lists**: when some kinds of object are freed (floats, tuples of small sizes, frames), CPython keeps the memory to reuse for the next one instead of returning it. That, plus the memory allocator reusing blocks, is why a new object often gets the **same `id`** as one just freed.
- **Immortal objects** (Python 3.12, PEP 683): `None`, `True`, `False`, small ints and other permanent objects have a special reference count that never changes, so threads and processes can share them without updating counts. `sys.getrefcount(None)` shows a huge, meaningless number.
- **pymalloc**: small objects (up to 512 bytes) come from CPython's own allocator, which carves pages into blocks of fixed sizes, faster than the system's `malloc` for millions of tiny objects.

On your own computer you can peek at a real header with `ctypes` (this reads raw memory, so never write through it):

```
import ctypes, sys

class PyObjectHeader(ctypes.Structure):
    _fields_ = [("ob_refcnt", ctypes.c_ssize_t), ("ob_type", ctypes.c_void_p)]

data = []
header = PyObjectHeader.from_address(id(data))
print(header.ob_refcnt, sys.getrefcount(data) - 1)      # equal (in the normal build)
print(header.ob_type == id(list))                        # the type pointer
```

## Practice

## Quiz

## Daily challenge

## Before you go

Find `Include/object.h` in the CPython source on GitHub and look for `struct _object`: that's today's header. Tomorrow: the import system, from `sys.modules` to writing your own importer.